In [1]:
import torch
import numpy as np
import random
import pandas as pd
from transformers import (
    set_seed,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from peft import get_peft_model, LoraConfig, TaskType

In [2]:
MODEL_NAME = "cointegrated/rubert-tiny2"
DATASET_NAME = "blinoff/kinopoisk"
SEED = 42

In [3]:
def fix_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)

fix_all_seeds(SEED)

### 1. Выбор набора данных

Для задачи классификации текстов я выбрал датасет blinoff/kinopoisk.

Обоснование: Это сбалансированный набор отзывов на фильмы на русском языке. Задача — определить тональность отзыва (Positive, Negative, Neutral).

Предобработка: Исходный датасет содержит только train сплит, поэтому я использую train_test_split для выделения тестовой выборки (15%), чтобы честно оценивать качество моделей. Метки классов (grade3) были переведены в числовой формат.

### 2. Выбор архитектуры модели

В качестве базовой модели выбрана cointegrated/rubert-tiny2.

Обоснование:

Язык: Модель предобучена на русском языке, что критично для датасета Кинопоиска.

Эффективность: Это дистиллированная версия BERT. Она очень компактная (~29M параметров), что позволяет быстро проводить эксперименты и обучение даже на слабых GPU (например, в Colab), при этом показывая достойное качество для простых задач классификации.

In [4]:
raw_dataset = load_dataset(DATASET_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
dataset = raw_dataset['train'].train_test_split(test_size=0.15, seed=SEED)
dataset


DatasetDict({
    train: Dataset({
        features: ['part', 'movie_name', 'review_id', 'author', 'date', 'title', 'grade3', 'grade10', 'content'],
        num_rows: 31102
    })
    test: Dataset({
        features: ['part', 'movie_name', 'review_id', 'author', 'date', 'title', 'grade3', 'grade10', 'content'],
        num_rows: 5489
    })
})

In [6]:
label2id = {'Good': 1, 'Bad': 0, 'Neutral': 2}
id2label = {1: 'Good', 0: 'Bad', 2: 'Neutral'}
NUM_LABELS = 3

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [8]:
def preprocess_function(examples):
    tokenized = tokenizer(
        examples['content'],
        truncation=True,
        max_length=512,
        padding=False
    )
    tokenized['labels'] = [label2id[g] for g in examples['grade3']]
    return tokenized


In [9]:
encoded_dataset = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/31102 [00:00<?, ? examples/s]

Map:   0%|          | 0/5489 [00:00<?, ? examples/s]

In [10]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1': f1}

In [11]:
results_table = []

In [12]:
import wandb
wandb.init(mode="disabled")


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


### 3. Оценка качества as-is (Zero-shot)

Проводим оценку модели сразу после инициализации, без обучения.

Ожидание: Поскольку классификационная голова инициализируется случайными весами (так как num_labels=3 не совпадает с предобученной головой, если она была).

Метрики: Используем Accuracy и F1-score (weighted), так как классы могут быть несбалансированы.

In [16]:
model_asis = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

trainer_asis = Trainer(
    model=model_asis,
    eval_dataset=encoded_dataset['test'],
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

metrics_asis = trainer_asis.evaluate()
results_table.append({
    "Method": "As-is (Random Init)",
    "Accuracy": metrics_asis['eval_accuracy'],
    "F1": metrics_asis['eval_f1'],
    "Time (sec)": 0
})

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-962880490.py:8: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_asis = Trainer(


**Вывод по As-is:**
Модель показала качество на уровне ~14% (Accuracy). Это ожидаемо: классификационная голова была инициализирована случайным шумом, и без обучения модель не может сопоставлять свои знания с нашими метками классов.


In [17]:
model_full = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id
)

args_full = TrainingArguments(
    output_dir="./results_full",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    seed=SEED,
    report_to="none"
)

trainer_full = Trainer(
    model=model_full,
    args=args_full,
    train_dataset=encoded_dataset['train'],
    eval_dataset=encoded_dataset['test'],
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

train_res_full = trainer_full.train()
metrics_full = trainer_full.evaluate()

results_table.append({
    "Method": "Full Finetuning",
    "Accuracy": metrics_full['eval_accuracy'],
    "F1": metrics_full['eval_f1'],
    "Time (sec)": train_res_full.metrics['train_runtime']
})


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3105866129.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_full = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.585900,0.396015,0.846967,0.817852
2,0.390400,0.381664,0.850246,0.832916
3,0.357700,0.383280,0.848971,0.834478


**Вывод по Full Finetuning:**
Разморозка всех весов дала резкий скачок качества — Accuracy достигла **~84.9%**.


In [18]:
model_lp = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id
)

for param in model_lp.bert.parameters():
    param.requires_grad = False


args_lp = TrainingArguments(
    output_dir="./results_lp",
    learning_rate=1e-3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    seed=SEED,
    report_to="none"
)

trainer_lp = Trainer(
    model=model_lp,
    args=args_lp,
    train_dataset=encoded_dataset['train'],
    eval_dataset=encoded_dataset['test'],
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

train_res_lp = trainer_lp.train()
metrics_lp = trainer_lp.evaluate()

results_table.append({
    "Method": "Linear Probing",
    "Accuracy": metrics_lp['eval_accuracy'],
    "F1": metrics_lp['eval_f1'],
    "Time (sec)": train_res_lp.metrics['train_runtime']
})

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1348852563.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_lp = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.685600,0.584650,0.769539,0.695804
2,0.588700,0.554166,0.779559,0.714171
3,0.566500,0.544262,0.780652,0.715619
4,0.562000,0.538804,0.787029,0.726604
5,0.551400,0.537463,0.787575,0.728368


**Вывод по Linear Probing:**
Обучение только головы дало Accuracy **~78.8%** (на ~6% хуже полного файнтьюна).

In [13]:
model_lora_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id
)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1
)

model_lora = get_peft_model(model_lora_base, peft_config)
model_lora.print_trainable_parameters()

args_lora = TrainingArguments(
    output_dir="./results_lora",
    learning_rate=1e-3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    seed=SEED,
    report_to="none"
)

trainer_lora = Trainer(
    model=model_lora,
    args=args_lora,
    train_dataset=encoded_dataset['train'],
    eval_dataset=encoded_dataset['test'],
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

train_res_lora = trainer_lora.train()
metrics_lora = trainer_lora.evaluate()

results_table.append({
    "Method": "LoRA",
    "Accuracy": metrics_lora['eval_accuracy'],
    "F1": metrics_lora['eval_f1'],
    "Time (sec)": train_res_lora.metrics['train_runtime']
})

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 30,891 || all params: 29,225,598 || trainable%: 0.1057


/tmp/ipython-input-994687605.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_lora = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.580600,0.430723,0.827473,0.808041
2,0.416300,0.386662,0.846056,0.821201
3,0.393600,0.388036,0.845327,0.820151
4,0.382400,0.382281,0.846238,0.828363
5,0.369500,0.380410,0.848788,0.828028


**Вывод по LoRA:**
Метод LoRA (обучение <1% параметров) позволил получить Accuracy **~84.9%**, что практически идентично полному файнтьюну. Мы получили SOTA-качество для этой модели, значительно сэкономив на количестве обучаемых параметров.


In [22]:
df_results = pd.DataFrame(results_table)
df_results

,Method,Accuracy,F1,Time (sec)
0,LoRA,0.848788,0.828028,1113.2139
1,As-is (Random Init),0.141738,0.059261,0.0000
2,Full Finetuning,0.848971,0.834478,699.2086
3,Linear Probing,0.787575,0.728368,454.2359


### Итоговые выводы

1.  **Качество:** `Full Finetuning` и `LoRA` показали одинаково высокий результат (85%). `Linear Probing` заметно отстает (~79%).
2.  **Эффективность:** LoRA доказала свою эффективность, позволив достичь качества полной модели при обучении мизерного количества параметров.
3.  **Рекомендация:** Для данной задачи оптимальным выбором является **LoRA**, так как она сочетает максимальное качество и гибкость (легкие веса адаптеров).
